# Generate All Figures

Set `RUN` and `WINDOW` in the config cell, then run cells top to bottom (or any cell individually — every figure cell reloads its own data and doesn't depend on other figure cells having run first).

**Notes:**
- Most quantitative figures (timeseries, rank histogram, spread-skill, PSD, Taylor diagram, CRPS, ...) load from the already-computed 4-window-pooled CSVs in `evaluation/saved_figs/<RUN>/_4window_aggregate/` — fast (sub-second), and that pooled CSV *is* what the paper's own tables/figures are built from.
- The spatial-map figures (domain overview, ensemble/error example fields, pattern-corr / bias-std-ratio / anomaly-corr maps, PIOMAS spatial snapshot). Those cells reload `eval_data/fields.npz` for one selected training `WINDOW` (several GB, tens of seconds) instead. This is a real limitation of what the pipeline saves today, not a shortcut taken here — noted again at each such cell.
- The 4-window-aggregate pipeline pools 05/06/10's *maps* by literally averaging the pixel maps from all 4 windows (see `build_4window_aggregate_figures.py`); the map cells below use a single representative `WINDOW` instead (matching sections 3/4/12's own convention in `run_daily_eval_batch.py`), so they'll differ slightly from the paper's own pooled map PNGs if you inspect those directly.

In [1]:
import os
import glob
import json
import pickle
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cartopy.crs as ccrs
from matplotlib.colors import TwoSlopeNorm

BASE = "/glade/work/skygale/projects/SeaIceDownscaling/Version6"
sys.path.insert(0, os.path.join(BASE, "training"))
sys.path.insert(0, os.path.join(BASE, "evaluation"))
import run_daily_eval_batch as rdeb  # noqa: E402
import functions_engressnet as fe  # noqa: E402

# ---- Choose run here ----
RUN = "MESA_noise_cascade_avg"   # a subdirectory name under results/ and evaluation/saved_figs/
WINDOW = "2015-2020"             # training-window substring; only used by the raw-npz (map) cells


def resolve_run_window_dir(base_subdir, run, window):
    """Find the single results/ or evaluation/saved_figs/ subfolder for (run, window). Folder names
    embed a PBS job id (e.g. MESA_noisecasc_avg_2015-2020_2021_5956149.casper-pbs), so this
    globs on the window substring rather than requiring the exact folder name."""
    pattern = os.path.join(BASE, base_subdir, run, f"*{window}*")
    matches = sorted(d for d in glob.glob(pattern) if os.path.isdir(d))
    if len(matches) != 1:
        raise FileNotFoundError(f"Expected exactly one match for {pattern!r}, found {len(matches)}: {matches}")
    return matches[0]


def agg_dir(run):
    return os.path.join(BASE, "evaluation", "saved_figs", run, "_4window_aggregate")


RESULTS_DIR = resolve_run_window_dir("results", RUN, WINDOW)
SAVED_FIGS_WINDOW_DIR = resolve_run_window_dir("evaluation/saved_figs", RUN, WINDOW)
AGG_DIR = agg_dir(RUN)
print("RUN            :", RUN)
print("WINDOW         :", WINDOW)
print("RESULTS_DIR    :", RESULTS_DIR)
print("SAVED_FIGS_DIR :", SAVED_FIGS_WINDOW_DIR)
print("AGG_DIR        :", AGG_DIR, "exists:", os.path.isdir(AGG_DIR))


/glade/work/skygale/conda-envs/downscaling_env/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


RUN            : MESA_noise_cascade_avg
WINDOW         : 2015-2020
RESULTS_DIR    : /glade/work/skygale/projects/SeaIceDownscaling/Version6/results/MESA_noise_cascade_avg/MESA_noisecasc_avg_2015-2020_2021_5956149.casper-pbs
SAVED_FIGS_DIR : /glade/work/skygale/projects/SeaIceDownscaling/Version6/saved_figs/MESA_noise_cascade_avg/MESA_noisecasc_avg_2015-2020_2021_5956149.casper-pbs
AGG_DIR        : /glade/work/skygale/projects/SeaIceDownscaling/Version6/saved_figs/MESA_noise_cascade_avg/_4window_aggregate exists: True


## 00 — Domain / mask overview  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
use_patches = meta["use_patches"]

with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
geo0 = tile_geometry[0] if not use_patches else None

fields = np.load(os.path.join(edir, "fields.npz"))
land_mask = fields["land_mask"]
hlat, hlon = fields["hlat"], fields["hlon"]

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)

fig = plt.figure(figsize=(6, 6))
ax = fig.add_subplot(1, 1, 1, projection=proj)
cf = ax.pcolormesh(hlon, hlat, land_mask[0, 0], cmap="Greys", vmin=0, vmax=1, shading="auto", transform=ccrs.PlateCarree())
if geo0 is not None:
    ax.plot(
        [geo0["target_lon"].min(), geo0["target_lon"].max(), geo0["target_lon"].max(), geo0["target_lon"].min(), geo0["target_lon"].min()],
        [geo0["target_lat"].min(), geo0["target_lat"].min(), geo0["target_lat"].max(), geo0["target_lat"].max(), geo0["target_lat"].min()],
        color="tab:blue", linewidth=1.5, transform=ccrs.PlateCarree(), label="test sub-domain",
    )
rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points)
ax.set_title("Land mask + candidate points" + (" + test sub-domain" if geo0 is not None else ""))
fig.colorbar(cf, ax=ax, shrink=0.7, label="1 = land")
plt.tight_layout()
plt.show()


## 00b — Pattern-corr check table  *(per-window CSV — no pooled version exists)*

In [ ]:
df_00b = pd.read_csv(os.path.join(SAVED_FIGS_WINDOW_DIR, "00b_pattern_corr_check.csv"), index_col=0)
df_00b


## 01 — Domain-mean SIT timeseries  *(pooled CSV)*

In [ ]:
ts_df = pd.read_csv(os.path.join(AGG_DIR, "01_domain_mean_sit_timeseries_data_4window.csv"), index_col=0, parse_dates=["time"])

ts_plot_cols = ["truth", "stochastic_unet_mean", "deterministic_unet", "bilinear", "piomas"]
ts_plot_df = ts_df.copy()
if rdeb.ROLLING_DAYS_TIMESERIES:
    win = rdeb.rolling_window_samples(ts_plot_df["time"], rdeb.ROLLING_DAYS_TIMESERIES)
    ts_plot_df[ts_plot_cols] = ts_plot_df[ts_plot_cols].rolling(win, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
for label, col, color, ls in [
    ("Truth", "truth", "black", "-"), ("Stochastic UNet Mean", "stochastic_unet_mean", "tab:blue", "-"),
    ("Deterministic UNet", "deterministic_unet", "tab:orange", "-"), ("Bilinear", "bilinear", "tab:green", "--"),
    ("PIOMAS (obs)", "piomas", "tab:red", ":"),
]:
    if ts_plot_df[col].notna().sum() == 0:
        continue
    ax.plot(ts_plot_df["time"], ts_plot_df[col], label=label, color=color, ls=ls, lw=2 if col == "truth" else 1.5)
ax.set_ylabel("Domain-mean SIT (m)"); ax.set_xlabel("Time"); ax.legend(fontsize=9)
fig.autofmt_xdate(); plt.tight_layout()
plt.show()


## 01b — Domain-mean pattern-correlation timeseries  *(pooled CSV)*

In [ ]:
pc_df = pd.read_csv(os.path.join(AGG_DIR, "01b_domain_mean_pattern_corr_timeseries_data_4window.csv"), index_col=0, parse_dates=["time"])

pc_plot_cols = ["bilinear", "piomas", "deterministic_unet", "stochastic_unet_mean"]
pc_plot_df = pc_df.copy()
if rdeb.ROLLING_DAYS_TIMESERIES:
    win = rdeb.rolling_window_samples(pc_plot_df["time"], rdeb.ROLLING_DAYS_TIMESERIES)
    pc_plot_df[pc_plot_cols] = pc_plot_df[pc_plot_cols].rolling(win, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
for label, col, color, ls in [
    ("Stochastic UNet Mean", "stochastic_unet_mean", "tab:blue", "-"),
    ("Deterministic UNet", "deterministic_unet", "tab:orange", "-"),
    ("Bilinear", "bilinear", "tab:green", "--"),
    ("PIOMAS (obs)", "piomas", "tab:red", ":"),
]:
    if pc_plot_df[col].notna().sum() == 0:
        continue
    ax.plot(pc_plot_df["time"], pc_plot_df[col], label=label, color=color, ls=ls, linewidth=1.5)
ax.axhline(1.0, color="black", linewidth=0.8, linestyle=":", alpha=0.5)
ax.set_ylabel("Domain-mean pattern correlation vs. Truth, pooled across 4 windows")
ax.set_xlabel("Time"); ax.set_ylim(top=1.02); ax.legend(fontsize=9)
fig.autofmt_xdate(); plt.tight_layout()
plt.show()

## 02 — Candidate-point timeseries  *(pooled CSV, multi-panel)*

In [ ]:
point_df = pd.read_csv(os.path.join(AGG_DIR, "02_candidate_point_timeseries_data_4window.csv"), index_col=0, parse_dates=["time"])

points_to_plot = list(dict.fromkeys(point_df["point"]))  # preserve first-seen order
fig, axs = plt.subplots(len(points_to_plot), 1, figsize=(10, 2.6 * len(points_to_plot)), sharex=True)
if len(points_to_plot) == 1:
    axs = [axs]
for ax, point_name in zip(axs, points_to_plot):
    sub = point_df[point_df["point"] == point_name]
    dist_km = sub["dist_km"].iloc[0] if len(sub) else np.nan
    for label, col, color, ls, lw in [
        ("Truth", "truth", "black", "-", 1.8), ("Stochastic UNet Mean", "stochastic_unet_mean", "tab:blue", "-", 1.2),
        ("Deterministic UNet", "deterministic_unet", "tab:orange", "-", 1.2),
        ("Bilinear", "bilinear", "tab:green", "--", 1.2), ("PIOMAS (obs)", "piomas", "tab:red", ":", 1.2),
    ]:
        m = sub[sub["method"] == col]
        if m["value"].notna().sum() == 0:
            continue
        y = m["value"]
        if rdeb.ROLLING_DAYS_TIMESERIES:
            y = y.rolling(rdeb.rolling_window_samples(m["time"], rdeb.ROLLING_DAYS_TIMESERIES), center=True, min_periods=1).mean()
        ax.plot(m["time"], y, label=label, color=color, linestyle=ls, linewidth=lw)
    ax.set_title(f"{point_name} (nearest valid ocean grid cell ~{dist_km:.1f} km away), pooled across 4 windows", fontsize=10)
    ax.set_ylabel("SIT (m)")
    if point_name == "Kivalina":
        ax.legend(fontsize=8, loc="upper left", frameon=False)
axs[-1].set_xlabel("Time")
fig.autofmt_xdate(); plt.tight_layout()
plt.show()

## 03 — Ensemble figure  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
X_test_sit_phys = fields["X_test_sit_phys"]; Y_base_phys = fields["Y_base_phys"]
Y_pred_det_phys = fields["Y_pred_det_phys"]; preds_all_phys = fields["preds_all_phys"]
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
test_tile_ids = fields["test_tile_ids"]

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

N_SAMPLES = min(3, Y_test_phys.shape[0])
MIN_ICE_THICKNESS = 0.5
MIN_ICE_FRAC = 0.25
ice_frac = (Y_test_phys[:, 0] > MIN_ICE_THICKNESS).mean(axis=(1, 2))
valid_idxs = np.where(ice_frac > MIN_ICE_FRAC)[0]
SAMPLE_IDXS = np.random.default_rng(0).choice(valid_idxs, N_SAMPLES, replace=False)
MEMBER_IDX = min(4, preds_all_phys.shape[1] - 1)
VMIN, VMAX = 0, 3
CMAP = "Blues"

panel_titles = ["Low-Res Input", "Bilinear", "Deterministic", "One Member", "Ensemble Mean", "High-Res Truth"]
fig, axs = plt.subplots(len(SAMPLE_IDXS), 6, figsize=(18, 3.3 * len(SAMPLE_IDXS)), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
if len(SAMPLE_IDXS) == 1:
    axs = axs[None, :]
for row, idx in enumerate(SAMPLE_IDXS):
    geo = tile_geometry[int(test_tile_ids[idx])]
    ctx_lon, ctx_lat = geo["context_lon"], geo["context_lat"]
    tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
    fields_row = [
        X_test_sit_phys[idx, 0], Y_base_phys[idx, 0], Y_pred_det_phys[idx, 0],
        preds_all_phys[idx, MEMBER_IDX, 0], Y_pred_phys[idx, 0], Y_test_phys[idx, 0],
    ]
    lons = [ctx_lon, tgt_lon, tgt_lon, tgt_lon, tgt_lon, tgt_lon]
    lats = [ctx_lat, tgt_lat, tgt_lat, tgt_lat, tgt_lat, tgt_lat]
    for col, (field, lon_, lat_) in enumerate(zip(fields_row, lons, lats)):
        ax = axs[row, col]
        im = ax.pcolormesh(lon_, lat_, field, transform=ccrs.PlateCarree(), cmap=CMAP, vmin=VMIN, vmax=VMAX, shading="auto")
        _style(ax, lon_, lat_)
        if row == 0:
            ax.set_title(panel_titles[col], fontsize=13)
    axs[row, 0].set_ylabel(f"Sample {row + 1}", fontsize=13)
cbar = fig.colorbar(im, ax=axs, aspect=30, shrink=0.8, pad=0.02)
cbar.set_label("Sea ice thickness (m)", fontsize=13)
plt.show()

## 04 — Error figure  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
Y_base_phys = fields["Y_base_phys"]; Y_pred_det_phys = fields["Y_pred_det_phys"]
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
test_tile_ids = fields["test_tile_ids"]

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

N_SAMPLES = min(3, Y_test_phys.shape[0])
MIN_ICE_THICKNESS = 0.5
MIN_ICE_FRAC = 0.25
ice_frac = (Y_test_phys[:, 0] > MIN_ICE_THICKNESS).mean(axis=(1, 2))
valid_idxs = np.where(ice_frac > MIN_ICE_FRAC)[0]
SAMPLE_IDXS = np.random.default_rng(0).choice(valid_idxs, N_SAMPLES, replace=False)

ERR_VMIN, ERR_VMAX = 0, 2
ERR_CMAP = "viridis"
panel_titles_err = ["Bilinear", "Deterministic", "Ensemble Mean"]
fig, axs = plt.subplots(len(SAMPLE_IDXS), 3, figsize=(9, 2.7 * len(SAMPLE_IDXS)), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
if len(SAMPLE_IDXS) == 1:
    axs = axs[None, :]
for row, idx in enumerate(SAMPLE_IDXS):
    geo = tile_geometry[int(test_tile_ids[idx])]
    tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
    truth = Y_test_phys[idx, 0]
    bilinear_ae = np.abs(Y_base_phys[idx, 0] - truth)
    det_ae = np.abs(Y_pred_det_phys[idx, 0] - truth)
    ens_ae = np.abs(Y_pred_phys[idx, 0] - truth)
    for col, field in enumerate([bilinear_ae, det_ae, ens_ae]):
        ax = axs[row, col]
        im = ax.pcolormesh(tgt_lon, tgt_lat, field, transform=ccrs.PlateCarree(), cmap=ERR_CMAP, vmin=ERR_VMIN, vmax=ERR_VMAX, shading="auto")
        _style(ax, tgt_lon, tgt_lat)
        if row == 0:
            ax.set_title(panel_titles_err[col], fontsize=13)
    axs[row, 0].set_ylabel(f"Sample {row + 1}", fontsize=13)
cbar = fig.colorbar(im, ax=axs, aspect=20, shrink=0.9, pad=0.02)
cbar.set_label("|Y - Truth| Absolute Error (m)", fontsize=12)
plt.show()

## 05 — Pattern-corr maps (March)  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
Y_base_phys = fields["Y_base_phys"]; Y_pred_det_phys = fields["Y_pred_det_phys"]
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
mask_test = fields["mask_test"]
sample_times_df = pd.read_csv(os.path.join(edir, "sample_times.csv"), parse_dates=["time"])

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

MONTH = 3
geo = tile_geometry[0]
tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
land_hw = mask_test[0, 0] > 0.5
month_idx = np.where(sample_times_df["time"].dt.month.values == MONTH)[0]

methods_maps = {}
for label, field in [("Bilinear", Y_base_phys), ("Deterministic UNet", Y_pred_det_phys), ("Stochastic UNet Mean", Y_pred_phys)]:
    corr_map = rdeb.temporal_pixel_corr(field, Y_test_phys, month_idx)
    methods_maps[label] = np.where(land_hw, np.nan, corr_map)

fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
for ax, (label, corr_map) in zip(axs, methods_maps.items()):
    im = ax.pcolormesh(tgt_lon, tgt_lat, corr_map, transform=ccrs.PlateCarree(), cmap="RdBu_r", vmin=0, vmax=1, shading="auto")
    _style(ax, tgt_lon, tgt_lat)
    ax.set_title(label, fontsize=13)
cbar = fig.colorbar(im, ax=axs, aspect=30, shrink=0.8, pad=0.02)
cbar.set_label(f"Temporal Pearson corr. vs. truth (month={MONTH}), window={WINDOW}", fontsize=12)
plt.show()

### 05 summary table  *(pooled CSV, domain-mean only — not the map itself)*

In [ ]:
pd.read_csv(os.path.join(AGG_DIR, "05_pattern_corr_summary_4window.csv"), index_col=0)


## 06 — Bias & std-ratio maps  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
Y_base_phys = fields["Y_base_phys"]; Y_pred_det_phys = fields["Y_pred_det_phys"]
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
mask_test = fields["mask_test"]

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

geo = tile_geometry[0]
tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
land_hw = mask_test[0, 0] > 0.5
BIAS_LIM = 0.5
STD_RATIO_LIM = (0, 2)
method_fields = {"Bilinear": Y_base_phys, "Deterministic UNet": Y_pred_det_phys, "Stochastic UNet Mean": Y_pred_phys}
truth_mean_hw = Y_test_phys[:, 0].mean(axis=0)
truth_std_hw = Y_test_phys[:, 0].std(axis=0)
bias_maps, std_ratio_maps = {}, {}
for label, field in method_fields.items():
    pred_mean_hw = field[:, 0].mean(axis=0)
    pred_std_hw = field[:, 0].std(axis=0)
    bias_maps[label] = np.where(land_hw, np.nan, pred_mean_hw - truth_mean_hw)
    safe_truth_std = np.where(truth_std_hw > 1e-8, truth_std_hw, np.nan)
    std_ratio_maps[label] = np.where(land_hw, np.nan, pred_std_hw / safe_truth_std)

fig, axs = plt.subplots(2, 3, figsize=(15, 9), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
for col, (label, bmap) in enumerate(bias_maps.items()):
    ax = axs[0, col]
    im0 = ax.pcolormesh(tgt_lon, tgt_lat, bmap, transform=ccrs.PlateCarree(), cmap="RdBu_r", vmin=-BIAS_LIM, vmax=BIAS_LIM, shading="auto")
    _style(ax, tgt_lon, tgt_lat)
    ax.set_title(f"{label}: time-mean bias", fontsize=12)
for col, (label, rmap) in enumerate(std_ratio_maps.items()):
    ax = axs[1, col]
    im1 = ax.pcolormesh(tgt_lon, tgt_lat, rmap, transform=ccrs.PlateCarree(), cmap="RdBu_r",
                         norm=TwoSlopeNorm(vcenter=1, vmin=STD_RATIO_LIM[0], vmax=STD_RATIO_LIM[1]), shading="auto")
    _style(ax, tgt_lon, tgt_lat)
    ax.set_title(f"{label}: std ratio (pred/truth)", fontsize=12)
fig.colorbar(im0, ax=axs[0, :], shrink=0.8, pad=0.02, label="Time-mean bias (m)")
fig.colorbar(im1, ax=axs[1, :], shrink=0.8, pad=0.02, label="Temporal std ratio")
plt.show()

### 06 summary table  *(pooled CSV, domain-mean only — not the map itself)*

In [ ]:
pd.read_csv(os.path.join(AGG_DIR, "06_bias_std_ratio_summary_4window.csv"), index_col=0)


## 08 — Domain-mean bias timeseries  *(derived from the 01 pooled CSV — same as the original script)*

In [ ]:
ts_df = pd.read_csv(os.path.join(AGG_DIR, "01_domain_mean_sit_timeseries_data_4window.csv"), index_col=0, parse_dates=["time"])
bias_df = ts_df.copy()
bias_df["bias_stochastic_unet_mean"] = bias_df["stochastic_unet_mean"] - bias_df["truth"]
bias_df["bias_deterministic_unet"] = bias_df["deterministic_unet"] - bias_df["truth"]
bias_df["bias_bilinear"] = bias_df["bilinear"] - bias_df["truth"]
bias_df["bias_piomas"] = bias_df["piomas"] - bias_df["truth"]
bias_cols = ["bias_stochastic_unet_mean", "bias_deterministic_unet", "bias_bilinear", "bias_piomas"]

bias_plot_df = bias_df.copy()
if rdeb.ROLLING_DAYS_BATCH:
    win = rdeb.rolling_window_samples(bias_plot_df["time"], rdeb.ROLLING_DAYS_BATCH)
    bias_plot_df[bias_cols] = bias_plot_df[bias_cols].rolling(win, center=True, min_periods=1).mean()

fig, ax = plt.subplots(figsize=(10, 4))
ax.axhline(0, color="black", linewidth=1, linestyle=":")
for label, col, color in [
    ("Stochastic UNet Mean", "bias_stochastic_unet_mean", "tab:blue"),
    ("Deterministic UNet", "bias_deterministic_unet", "tab:orange"),
    ("Bilinear", "bias_bilinear", "tab:green"),
    ("PIOMAS (obs)", "bias_piomas", "tab:red"),
]:
    if bias_plot_df[col].notna().sum() == 0:
        continue
    ax.plot(bias_plot_df["time"], bias_plot_df[col], label=label, color=color, linewidth=1.5)
ax.set_ylabel("Domain-mean bias (pred - truth, m)"); ax.set_xlabel("Time"); ax.legend(fontsize=9)
fig.autofmt_xdate(); plt.tight_layout()
plt.show()

bias_df[bias_cols].agg(["mean", "std"]).round(4)


## 09 — Seasonal climatology  *(derived from the 01 pooled CSV — same as the original script)*

In [ ]:
ts_df = pd.read_csv(os.path.join(AGG_DIR, "01_domain_mean_sit_timeseries_data_4window.csv"), index_col=0, parse_dates=["time"])
clim_df = ts_df.copy()
clim_df["month"] = clim_df["time"].dt.month
monthly_clim = clim_df.groupby("month")[["truth", "stochastic_unet_mean", "deterministic_unet", "bilinear", "piomas"]].mean()

fig, ax = plt.subplots(figsize=(8, 4.5))
for col, label, color, ls in [
    ("truth", "Truth", "black", "-"), ("stochastic_unet_mean", "Stochastic UNet Mean", "tab:blue", "-"),
    ("deterministic_unet", "Deterministic UNet", "tab:orange", "-"), ("bilinear", "Bilinear", "tab:green", "--"),
    ("piomas", "PIOMAS (obs)", "tab:red", ":"),
]:
    if monthly_clim[col].notna().sum() == 0:
        continue
    ax.plot(monthly_clim.index, monthly_clim[col], label=label, color=color, linestyle=ls, linewidth=1.5, marker="o", markersize=3)
ax.set_xticks(range(1, 13)); ax.set_xlabel("Month")
ax.set_ylabel("Domain-mean SIT climatology (m), pooled across 4 windows")
ax.legend(fontsize=9); plt.tight_layout()
plt.show()

(monthly_clim.max() - monthly_clim.min()).rename("Seasonal amplitude (m)").round(4).to_frame()


## 10 — Anomaly-correlation maps  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
Y_base_phys = fields["Y_base_phys"]; Y_pred_det_phys = fields["Y_pred_det_phys"]
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
mask_test = fields["mask_test"]
sample_times_df = pd.read_csv(os.path.join(edir, "sample_times.csv"), parse_dates=["time"])

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

geo = tile_geometry[0]
tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
land_hw = mask_test[0, 0] > 0.5
method_fields = {"Bilinear": Y_base_phys, "Deterministic UNet": Y_pred_det_phys, "Stochastic UNet Mean": Y_pred_phys}

months = sample_times_df["time"].dt.month.values
all_idx = np.arange(len(months))
truth_anom = rdeb.remove_monthly_climatology(Y_test_phys, months)
acc_maps = {}
for label, field in method_fields.items():
    pred_anom = rdeb.remove_monthly_climatology(field, months)
    acc_maps[label] = np.where(land_hw, np.nan, rdeb.temporal_pixel_corr(pred_anom, truth_anom, all_idx))

fig, axs = plt.subplots(1, 3, figsize=(15, 5), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
for ax, (label, amap) in zip(axs, acc_maps.items()):
    im = ax.pcolormesh(tgt_lon, tgt_lat, amap, transform=ccrs.PlateCarree(), cmap="RdBu_r", vmin=-1, vmax=1, shading="auto")
    _style(ax, tgt_lon, tgt_lat)
    ax.set_title(label, fontsize=13)
cbar = fig.colorbar(im, ax=axs, aspect=30, shrink=0.8, pad=0.02)
cbar.set_label(f"Anomaly correlation (deseasonalized), all months, window={WINDOW}", fontsize=12)
plt.show()

### 10 summary table  *(pooled CSV, domain-mean only — not the map itself)*

In [ ]:
pd.read_csv(os.path.join(AGG_DIR, "10_anomaly_corr_summary_4window.csv"), index_col=0)


## 11 — Taylor diagram  *(pooled CSV)*

In [ ]:
taylor_df = pd.read_csv(os.path.join(AGG_DIR, "11_taylor_stats_4window.csv"), index_col=0)
taylor_stats = {row["Method"]: (row["Correlation (R)"], row["Std Ratio"], row["Centered RMSE (m)"])
                for _, row in taylor_df.iterrows()}

def rms_circle(center, radius, theta_lim=(0, np.pi / 2), n=200):
    t = np.linspace(0, 2 * np.pi, n)
    x, y = center[0] + radius * np.cos(t), center[1] + radius * np.sin(t)
    theta, r = np.arctan2(y, x), np.sqrt(x ** 2 + y ** 2)
    valid = (theta >= theta_lim[0]) & (theta <= theta_lim[1])
    return theta[valid], r[valid]

max_std = max(sr for _, sr, _ in taylor_stats.values())
r_lim = max(1.3, max_std * 1.2)
fig = plt.figure(figsize=(6.5, 6.5))
ax = fig.add_subplot(111, polar=True)
ax.set_thetamin(0); ax.set_thetamax(90)
ax.set_theta_zero_location("E"); ax.set_theta_direction(1)
ax.set_ylim(0, r_lim)
corr_ticks = np.array([0, 0.2, 0.4, 0.6, 0.8, 0.9, 0.95, 0.99, 1.0])
ax.set_xticks(np.arccos(corr_ticks)); ax.set_xticklabels([str(c) for c in corr_ticks])
ax.set_xlabel("Correlation", labelpad=10)
ax.set_ylabel("Normalized std. dev. (relative to Truth)", labelpad=30)
for rms in np.arange(0.5, r_lim, 0.5):
    th, r = rms_circle((1, 0), rms)
    ax.plot(th, r, color="gray", linestyle=":", linewidth=0.8)
std_circle_theta = np.linspace(0, np.pi / 2, 100)
ax.plot(std_circle_theta, np.ones_like(std_circle_theta), color="black", linestyle="--", linewidth=0.8)
ax.plot([0], [1], marker="*", color="black", markersize=16, linestyle="none", label="Truth (reference)")
colors = {"Bilinear": "tab:green", "Deterministic UNet": "tab:orange", "Stochastic UNet Mean": "tab:blue"}
for label, (R, std_ratio, crmse) in taylor_stats.items():
    ax.plot([np.arccos(np.clip(R, -1, 1))], [std_ratio], marker="o", markersize=10,
            color=colors.get(label, "tab:red"), linestyle="none", label=label)
ax.legend(loc="upper right", bbox_to_anchor=(1.35, 1.1), fontsize=9)
plt.tight_layout()
plt.show()

## 12 — PIOMAS metrics table  *(pooled CSV)*

In [ ]:
pd.read_csv(os.path.join(AGG_DIR, "12_piomas_metrics_4window.csv"), index_col=0)


### 12 — PIOMAS spatial snapshot  *(raw npz, single window)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
with open(os.path.join(edir, "meta.json")) as f:
    meta = json.load(f)
bbox = meta["bbox"]
candidate_points = {**meta["candidate_points"], "Point Hope": {"lat": rdeb.POINT_HOPE_LAT, "lon": rdeb.POINT_HOPE_LON_360}}
with open(os.path.join(edir, "tile_geometry.pkl"), "rb") as f:
    tile_geometry = pickle.load(f)
fields = np.load(os.path.join(edir, "fields.npz"))
Y_pred_phys = fields["Y_pred_phys"]; Y_test_phys = fields["Y_test_phys"]
sample_times_df = pd.read_csv(os.path.join(edir, "sample_times.csv"), parse_dates=["time"])

proj, boundary_path, central_lon = rdeb.make_polar_proj(bbox)
def _style(ax, lon_=None, lat_=None):
    rdeb.style_polar_ax(ax, proj, boundary_path, bbox, candidate_points, lon_, lat_)

geo = tile_geometry[0]
tgt_lon, tgt_lat = geo["target_lon"], geo["target_lat"]
piomas_regridded_phys = rdeb.build_piomas_regridded(tgt_lat, tgt_lon, sample_times_df["time"])
piomas_valid = ~np.isnan(piomas_regridded_phys[:, 0]).all(axis=(1, 2))

if piomas_valid.sum() == 0:
    print("No test samples in this window overlap PIOMAS's record -- nothing to plot.")
else:
    valid_idxs = np.where(piomas_valid)[0]
    ice_valid = valid_idxs[(Y_test_phys[valid_idxs, 0] > 0.5).mean(axis=(1, 2)) > 0.25]
    piomas_sample_idx = int(ice_valid[0]) if len(ice_valid) else int(valid_idxs[0])
    panel_fields = [Y_test_phys[piomas_sample_idx, 0], Y_pred_phys[piomas_sample_idx, 0], piomas_regridded_phys[piomas_sample_idx, 0]]
    panel_titles_piomas = ["Truth", "Ensemble Mean", "PIOMAS (obs, regridded)"]
    fig, axs = plt.subplots(1, 3, figsize=(13.5, 4.5), constrained_layout=True, dpi=150, subplot_kw={"projection": proj})
    for ax, field, title in zip(axs, panel_fields, panel_titles_piomas):
        im = ax.pcolormesh(tgt_lon, tgt_lat, field, transform=ccrs.PlateCarree(), cmap="Blues", vmin=0, vmax=3, shading="auto")
        _style(ax, tgt_lon, tgt_lat)
        ax.set_title(title, fontsize=13)
    sample_time = sample_times_df["time"].iloc[piomas_sample_idx]
    fig.suptitle(f"Sample {piomas_sample_idx} ({sample_time:%Y-%m})", fontsize=12)
    cbar = fig.colorbar(im, ax=axs, aspect=30, shrink=0.8, pad=0.02)
    cbar.set_label("Sea ice thickness (m)", fontsize=13)
    plt.show()

## 13 — PSD comparison  *(pooled CSV)*

In [ ]:
psd_df = pd.read_csv(os.path.join(AGG_DIR, "13_psd_data_4window.csv"), index_col=0)
method_cols = [c for c in psd_df.columns if c not in ("wavenumber_cycles_per_km", "wavelength_km")]

fig, ax = plt.subplots(figsize=(7, 5.5))
wavenumber = psd_df["wavenumber_cycles_per_km"].values
for c in method_cols:
    valid = (wavenumber > 0) & (psd_df[c] > 0)
    ax.plot(wavenumber[valid], psd_df[c][valid], label=c, linewidth=1.8)
ax.set_xscale("log"); ax.set_yscale("log")
ax.set_xlabel("Wavenumber (cycles/km)"); ax.set_ylabel("Isotropic PSD (m$^2$, arb. spectral units)")
ax.set_title("Domain-wide PSD, averaged across 4 train windows (test=2021)")
ax.legend(fontsize=9); ax.grid(True, which="both", alpha=0.3)
plt.tight_layout()
plt.show()

## 13b — Per-member PSD spread  *(pooled CSV)*

In [ ]:
psdb_df = pd.read_csv(os.path.join(AGG_DIR, "13b_psd_per_member_spread_data_4window.csv"), index_col=0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 5.5))
wavenumber = psdb_df["wavenumber_cycles_per_km"].values
valid = (wavenumber > 0) & (psdb_df["Truth"] > 0)
ax1.plot(wavenumber[valid], psdb_df["Truth"][valid], label="Truth", color="black", linewidth=1.8)
ax1.fill_between(wavenumber[valid], psdb_df["Member p10"][valid], psdb_df["Member p90"][valid],
                  color="tab:blue", alpha=0.3, label="Stochastic UNet members (10th-90th pct)")
ax1.plot(wavenumber[valid], psdb_df["Member mean"][valid], color="tab:blue", linewidth=1.2, linestyle="--",
         label="Stochastic UNet member mean")
ax1.set_xscale("log"); ax1.set_yscale("log")
ax1.set_xlabel("Wavenumber (cycles/km)"); ax1.set_ylabel("Isotropic PSD (m$^2$, arb. spectral units)")
ax1.set_title("Per-member PSD spread vs. truth")
ax1.legend(fontsize=8); ax1.grid(True, which="both", alpha=0.3)

ax2.plot(wavenumber[valid], psdb_df["Relative member spread pct"][valid], color="tab:blue", linewidth=1.8)
ax2.set_xscale("log")
ax2.set_xlabel("Wavenumber (cycles/km)")
ax2.set_ylabel("Relative member spread,\n(p90 - p10) / mean (%)")
ax2.set_title("Inter-member spectral-texture spread\n(near-zero = members look nearly identical)")
ax2.grid(True, which="both", alpha=0.3)
fig.suptitle("Per-member PSD spread, averaged across 4 train windows", y=1.02)
plt.tight_layout()
plt.show()

## 14a — Rank histogram  *(pooled CSV)*

In [ ]:
rank_hist_df = pd.read_csv(os.path.join(AGG_DIR, "14_rank_histogram_data_4window.csv"), index_col=0)
K = len(rank_hist_df) - 1
expected_freq = rank_hist_df["expected_frequency"].iloc[0]

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 4.5))
ax1.bar(rank_hist_df["rank"], rank_hist_df["frequency"], width=0.9, color="tab:blue", alpha=0.85, label="Observed")
ax1.axhline(expected_freq, color="black", linestyle="--", linewidth=1.2, label="Perfect calibration (flat)")
ax1.set_xlabel(f"Rank of truth among {K} sorted ensemble members")
ax1.set_ylabel("Frequency")
ax1.set_title("Rank histogram, pooled across 4 train windows (test=2021)")
ax1.legend(fontsize=9, frameon=False)
ax2.bar(rank_hist_df["rank"], rank_hist_df["Calibration Score (1.0 = ideal)"], width=0.9, color="tab:green", alpha=0.85)
ax2.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="Perfect calibration")
ax2.set_xlabel(f"Rank of truth among {K} sorted ensemble members")
ax2.set_ylabel("Calibration score (min(ratio, 1/ratio))")
ax2.set_ylim(0, 1.05)
ax2.set_title("Per-rank calibration score, pooled")
ax2.legend(fontsize=9, frameon=False, loc="lower right")
plt.tight_layout()
plt.show()

## 14b — Reliability diagram  *(raw npz, single window — **no CSV form exists anywhere in the
pipeline**, pooled or per-window; `14_reliability_diagram_data*.csv` only ever stores the
per-threshold Brier score/climatological frequency, never the per-bin forecast/observed-frequency
curve the plot itself needs, so this always recomputes from the raw ensemble)*

In [ ]:
edir = os.path.join(RESULTS_DIR, "eval_data")
fields = np.load(os.path.join(edir, "fields.npz"))
preds_all_phys = fields["preds_all_phys"]; Y_test_phys = fields["Y_test_phys"]; mask_test = fields["mask_test"]

ocean_bool = mask_test[:, 0] <= 0.5
ens_hwk = np.moveaxis(preds_all_phys[:, :, 0], 1, -1)
ens_ocean = ens_hwk[ocean_bool]
truth_ocean = Y_test_phys[:, 0][ocean_bool]

RELIABILITY_THRESHOLDS = [0.15, 0.5, 1.0]
N_PROB_BINS = 10
bin_edges = np.linspace(0, 1, N_PROB_BINS + 1)
bin_centers = 0.5 * (bin_edges[:-1] + bin_edges[1:])
threshold_colors = ["tab:blue", "tab:orange", "tab:red"]

curves = {}
for thr in RELIABILITY_THRESHOLDS:
    p_forecast = (ens_ocean > thr).mean(axis=1)
    outcome = (truth_ocean > thr).astype(float)
    bin_idx = np.clip(np.digitize(p_forecast, bin_edges[1:-1]), 0, N_PROB_BINS - 1)
    mean_fc = np.full(N_PROB_BINS, np.nan)
    obs_freq = np.full(N_PROB_BINS, np.nan)
    counts = np.zeros(N_PROB_BINS, dtype=int)
    for b in range(N_PROB_BINS):
        sel = bin_idx == b
        counts[b] = sel.sum()
        if counts[b] > 0:
            mean_fc[b] = p_forecast[sel].mean()
            obs_freq[b] = outcome[sel].mean()
    curves[thr] = {"mean_forecast": mean_fc, "obs_freq": obs_freq, "count": counts}

fig, (ax_main, ax_hist) = plt.subplots(2, 1, figsize=(6.5, 7), gridspec_kw={"height_ratios": [3, 1]}, sharex=True)
ax_main.plot([0, 1], [0, 1], color="black", linestyle="--", linewidth=1, label="Perfectly reliable")
for thr, color in zip(RELIABILITY_THRESHOLDS, threshold_colors):
    c = curves[thr]
    valid = c["count"] > 0
    sizes = 20 + 200 * c["count"][valid] / c["count"].max()
    ax_main.plot(c["mean_forecast"][valid], c["obs_freq"][valid], color=color, linewidth=1.5, zorder=2)
    ax_main.scatter(c["mean_forecast"][valid], c["obs_freq"][valid], s=sizes, color=color, zorder=3, label=f"SIT > {thr} m")
    ax_hist.bar(bin_centers, c["count"] / c["count"].sum(), width=0.8 / N_PROB_BINS, color=color, alpha=0.5, label=f"SIT > {thr} m")
ax_main.set_ylabel("Observed frequency")
ax_main.set_xlim(0, 1); ax_main.set_ylim(0, 1)
ax_main.legend(fontsize=9, frameon=False, loc="upper left")
ax_main.set_title(f"Reliability diagram, window={WINDOW} (test=2021)")
ax_hist.set_xlabel("Forecast probability")
ax_hist.set_ylabel("Fraction of\nsamples")
ax_hist.legend(fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 15 — Spread-skill by regime  *(pooled CSV)*

In [ ]:
spread_skill_df = pd.read_csv(os.path.join(AGG_DIR, "15_spread_skill_by_regime_data_4window.csv"), index_col=0)
SIT_BIN_LABELS = spread_skill_df["SIT regime"].tolist()

x = np.arange(len(SIT_BIN_LABELS))
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.5))
width = 0.35
ax1.bar(x - width / 2, spread_skill_df["Mean ensemble spread (m)"], width, color="tab:blue", label="Ensemble spread (std)")
ax1.bar(x + width / 2, spread_skill_df["Mean |error| (m)"], width, color="tab:orange", label="|Ensemble mean - truth|")
ax1.set_xticks(x); ax1.set_xticklabels(SIT_BIN_LABELS, fontsize=8)
ax1.set_ylabel("m"); ax1.set_title("Spread vs. error by true-SIT regime, pooled across 4 windows")
ax1.legend(fontsize=9, frameon=False)
ax2.bar(x, spread_skill_df["Calibration Score (1.0 = ideal)"], color="tab:green", alpha=0.85)
ax2.axhline(1.0, color="black", linestyle="--", linewidth=1.2, label="Perfect calibration")
ax2.set_xticks(x); ax2.set_xticklabels(SIT_BIN_LABELS, fontsize=8)
ax2.set_ylabel("Calibration score (min(ratio, 1/ratio))"); ax2.set_ylim(0, 1.05)
ax2.set_title("Normalized calibration score by regime, pooled"); ax2.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

## 16 — CRPS by regime  *(pooled CSV)*

In [ ]:
crps_df = pd.read_csv(os.path.join(AGG_DIR, "16_crps_by_regime_data_4window.csv"), index_col=0)

fig, ax = plt.subplots(figsize=(7, 4.5))
plot_rows = crps_df.iloc[1:]
ax.bar(np.arange(len(plot_rows)), plot_rows["CRPS (m)"], color="tab:purple", alpha=0.85)
ax.axhline(crps_df.iloc[0]["CRPS (m)"], color="black", linestyle="--", linewidth=1.2,
           label=f"Domain-wide CRPS ({crps_df.iloc[0]['CRPS (m)']:.4f} m)")
ax.set_xticks(np.arange(len(plot_rows))); ax.set_xticklabels(plot_rows["SIT regime"], fontsize=8)
ax.set_ylabel("CRPS (m, lower = better)")
ax.set_title("CRPS by true-SIT regime, pooled across 4 train windows (test=2021)")
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

## Loss curve  *(results/, raw .npy — one array, not really a CSV-shaped thing)*

In [ ]:
loss_array = np.load(os.path.join(RESULTS_DIR, "loss_array.npy"))
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.plot(loss_array, color="tab:blue", linewidth=1.2)
ax.set_xlabel("Epoch"); ax.set_ylabel("Training loss (energy score)")
ax.set_title(f"Training loss curve -- {RUN}, window {WINDOW}")
plt.tight_layout()
plt.show()

## sit_timeseries  *(results/, CSV)*

In [ ]:
sit_df = pd.read_csv(os.path.join(RESULTS_DIR, "sit_timeseries.csv"), parse_dates=["time"])

fig, ax = plt.subplots(figsize=(10, 4))
for col, label, color, ls in [
    ("truth", "Truth", "black", "-"), ("stochastic_unet_mean", "Stochastic UNet Mean", "tab:blue", "-"),
    ("deterministic_unet", "Deterministic UNet", "tab:orange", "-"), ("bilinear", "Bilinear", "tab:green", "--"),
]:
    ax.plot(sit_df["time"], sit_df[col], label=label, color=color, linestyle=ls, linewidth=1.3)
ax.set_ylabel("SIT (m)"); ax.set_xlabel("Time"); ax.legend(fontsize=9)
fig.autofmt_xdate(); plt.tight_layout()
plt.show()

---
# Appendix / paper-level figures (not `RUN`-selector-dependent)

These are cross-run comparisons the manuscript builds once, from specific fixed runs, not from
whichever `RUN` you set above.


## Seasonal cycle (manuscript Fig. 7)  *(CSV — cross-run, fixed FOSI/MESA/PIOMAS comparison)*

In [ ]:
seasonal_csv = os.path.join(BASE, "figures", "piomas_fosi_mesa_seasonal_cycle_2021.csv")
seasonal_df = pd.read_csv(seasonal_csv, index_col=0)

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(seasonal_df.index, seasonal_df["PIOMAS (obs)"], color="black", linewidth=2.2, marker="o", markersize=4, label="PIOMAS (obs)")
ax.plot(seasonal_df.index, seasonal_df["FOSI truth"], color="tab:blue", linewidth=1.8, marker="o", markersize=3, label="FOSI (perfect-model truth)")
ax.plot(seasonal_df.index, seasonal_df["MESA truth"], color="tab:orange", linewidth=1.8, marker="o", markersize=3, label="MESACLIP (perfect-model truth)")
ax.set_xticks(range(1, 13))
ax.set_xlabel("Month"); ax.set_ylabel("Domain-mean sea ice thickness (m)")
ax.set_title("Seasonal cycle: PIOMAS vs. FOSI and MESACLIP, 2021")
ax.legend(fontsize=9, frameon=False)
plt.tight_layout()
plt.show()

## PIOMAS application (manuscript Fig. 8)  *(raw npz + candidate-point CSV — cross-run, fixed
`PIOMAS_obs_2021paper` run; no CSV form exists for the monthly maps or the daily ensemble-spread
band, so this always reloads raw fields)*

In [ ]:
piomas_app_matches = [
    d for d in sorted(glob.glob(os.path.join(BASE, "results", "PIOMAS_obs_2021paper", "*")))
    if os.path.isdir(d) and os.path.exists(os.path.join(d, "eval_data", "fields.npz"))
]
assert len(piomas_app_matches) == 1, piomas_app_matches
RUN_DIR_FIG8 = piomas_app_matches[0]
EVAL_DIR_FIG8 = os.path.join(RUN_DIR_FIG8, "eval_data")

fields8 = np.load(os.path.join(EVAL_DIR_FIG8, "fields.npz"))
Y_pred_phys8 = fields8["Y_pred_phys"][:, 0]
preds_all_phys8 = fields8["preds_all_phys"][:, :, 0]
mask_test8 = fields8["mask_test"]
with open(os.path.join(EVAL_DIR_FIG8, "tile_geometry.pkl"), "rb") as f:
    tile_geometry8 = pickle.load(f)
target_lat8 = np.asarray(tile_geometry8[0]["target_lat"])
target_lon8 = np.asarray(tile_geometry8[0]["target_lon"])
sample_times8 = pd.read_csv(os.path.join(EVAL_DIR_FIG8, "sample_times.csv"))
sample_times8["time"] = pd.to_datetime(sample_times8["time"])
sample_month8 = sample_times8["time"].dt.month.values
ocean_hw8 = mask_test8[0, 0] <= 0.5
point_df8 = pd.read_csv(os.path.join(EVAL_DIR_FIG8, "candidate_point_timeseries.csv"))

MONTHS = [1, 4, 7, 10]
MONTH_NAMES = {1: "January", 4: "April", 7: "July", 10: "October"}
plot_bbox = {"lon_min": -182, "lon_max": -151, "lat_min": 60, "lat_max": 75}
proj8, boundary_path8, central_lon8 = fe.make_polar_proj(plot_bbox)
lon2d, lat2d = np.meshgrid(target_lon8, target_lat8)

fig = plt.figure(figsize=(18, 13))
for i, month in enumerate(MONTHS):
    ax = fig.add_subplot(2, 4, i + 1, projection=proj8)
    fe.style_polar_ax(ax, proj8, boundary_path8, plot_bbox)
    day_idx = np.where(sample_month8 == month)[0]
    month_mean = np.where(ocean_hw8, Y_pred_phys8[day_idx].mean(axis=0), np.nan)
    pc = ax.pcolormesh(lon2d, lat2d, month_mean, transform=ccrs.PlateCarree(), cmap="viridis", vmin=0, vmax=2.5, shading="auto")
    ax.set_title(f"{MONTH_NAMES[month]} 2020", fontsize=12)
cax = fig.add_axes([0.25, 0.53, 0.5, 0.018])
fig.colorbar(pc, cax=cax, orientation="horizontal", label="Ensemble-mean SIT (m)")

for j, point_name in enumerate(["Kivalina", "Point Hope"]):
    ax = fig.add_subplot(2, 2, 3 + j)
    row = point_df8[(point_df8.point == point_name) & (point_df8.method == "stochastic_unet_mean")]
    iy, ix = int(row["grid_iy"].iloc[0]), int(row["grid_ix"].iloc[0])
    daily_mean = Y_pred_phys8[:, iy, ix]
    daily_spread = preds_all_phys8[:, :, iy, ix].std(axis=1)
    dates = sample_times8["time"].values
    ax.fill_between(dates, daily_mean - daily_spread, daily_mean + daily_spread, alpha=0.3, color="#2a78d6", label="\u00b11 ensemble std")
    ax.plot(dates, daily_mean, color="#2a78d6", linewidth=1.2, label="Ensemble mean")
    ax.set_title(f"{point_name}: daily downscaled SIT, 2020", fontsize=12)
    ax.set_ylabel("SIT (m)")
    ax.legend(loc="upper left", fontsize=9)
    ax.tick_params(axis="x", rotation=30)

fig.suptitle("Downscaled, daily, probabilistic coastal SIT, 2020\n(PIOMAS-driven input, MESA-warm-started FOSI-trained network)", y=0.98, fontsize=14)
fig.subplots_adjust(hspace=0.45, wspace=0.3)
plt.show()

## Sensitivity heatmap A1 (MESA)  *(CSV — exported from `make_sensitivity_heatmap.py`'s hardcoded
`mesa_rows` into `sensitivity/analysis/sensitivity_data_mesa.csv`; edit that CSV to add/adjust rows)*

In [ ]:
mesa_df = pd.read_csv(os.path.join(BASE, "sensitivity", "analysis", "sensitivity_data_mesa.csv"), index_col=0)

METRIC_DIRECTION = {
    "MAE": "lower", "RMSE": "lower", "Bias": "zero", "Grad MAE": "lower",
    "Pattern Corr": "higher", "SSIM": "higher", "IIEE": "lower",
    "Coastal MAE": "lower", "Coastal RMSE": "lower", "Spread/Error": "one",
}

def badness(series, direction):
    if direction == "lower":
        return series
    if direction == "higher":
        return -series
    if direction == "zero":
        return series.abs()
    if direction == "one":
        return (series - 1).abs()
    raise ValueError(direction)

def normalize_0_1(series):
    lo, hi = series.min(), series.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return series * 0.0
    return (series - lo) / (hi - lo)

def plot_badness_heatmap(value_df, title, figsize=None, cmap="Reds", fmt="{:.3f}"):
    badness_df = pd.DataFrame(index=value_df.index, columns=value_df.columns, dtype=float)
    for col in value_df.columns:
        direction = METRIC_DIRECTION.get(col, "lower")
        badness_df[col] = normalize_0_1(badness(value_df[col], direction))
    if figsize is None:
        figsize = (1.15 * len(value_df.columns) + 2, 0.5 * len(value_df.index) + 2)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(badness_df.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(value_df.columns))); ax.set_xticklabels(value_df.columns, rotation=40, ha="right")
    ax.set_yticks(range(len(value_df.index))); ax.set_yticklabels(value_df.index)
    for i in range(value_df.shape[0]):
        for j in range(value_df.shape[1]):
            v = value_df.values[i, j]
            if pd.isna(v):
                continue
            b = badness_df.values[i, j]
            text_color = "white" if b > 0.6 else "black"
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8, color=text_color)
    ax.set_title(title, fontsize=13)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks(np.arange(-0.5, len(value_df.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(value_df.index), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    fig.tight_layout()
    return fig, ax

fig1, ax1 = plot_badness_heatmap(
    mesa_df,
    "A1: Sensitivity tests, MESA -- configuration x metric\n"
    "color = relative badness within this table (lightest = best, darkest = worst per column); blank = not measured",
)
plt.show()

## Sensitivity heatmap A2 (FOSI)  *(CSV — exported from `make_sensitivity_heatmap.py`'s hardcoded
`fosi_rows` into `sensitivity/analysis/sensitivity_data_fosi.csv`; edit that CSV to add/adjust rows)*

In [ ]:
fosi_df = pd.read_csv(os.path.join(BASE, "sensitivity", "analysis", "sensitivity_data_fosi.csv"), index_col=0)

METRIC_DIRECTION = {
    "MAE": "lower", "RMSE": "lower", "Bias": "zero", "Grad MAE": "lower",
    "Pattern Corr": "higher", "SSIM": "higher", "IIEE": "lower",
    "Coastal MAE": "lower", "Coastal RMSE": "lower", "Spread/Error": "one",
}

def badness(series, direction):
    if direction == "lower":
        return series
    if direction == "higher":
        return -series
    if direction == "zero":
        return series.abs()
    if direction == "one":
        return (series - 1).abs()
    raise ValueError(direction)

def normalize_0_1(series):
    lo, hi = series.min(), series.max()
    if not np.isfinite(lo) or not np.isfinite(hi) or hi - lo < 1e-12:
        return series * 0.0
    return (series - lo) / (hi - lo)

def plot_badness_heatmap(value_df, title, figsize=None, cmap="Reds", fmt="{:.3f}"):
    badness_df = pd.DataFrame(index=value_df.index, columns=value_df.columns, dtype=float)
    for col in value_df.columns:
        direction = METRIC_DIRECTION.get(col, "lower")
        badness_df[col] = normalize_0_1(badness(value_df[col], direction))
    if figsize is None:
        figsize = (1.15 * len(value_df.columns) + 2, 0.5 * len(value_df.index) + 2)
    fig, ax = plt.subplots(figsize=figsize)
    im = ax.imshow(badness_df.values, cmap=cmap, vmin=0, vmax=1, aspect="auto")
    ax.set_xticks(range(len(value_df.columns))); ax.set_xticklabels(value_df.columns, rotation=40, ha="right")
    ax.set_yticks(range(len(value_df.index))); ax.set_yticklabels(value_df.index)
    for i in range(value_df.shape[0]):
        for j in range(value_df.shape[1]):
            v = value_df.values[i, j]
            if pd.isna(v):
                continue
            b = badness_df.values[i, j]
            text_color = "white" if b > 0.6 else "black"
            ax.text(j, i, fmt.format(v), ha="center", va="center", fontsize=8, color=text_color)
    ax.set_title(title, fontsize=13)
    for spine in ax.spines.values():
        spine.set_visible(False)
    ax.set_xticks(np.arange(-0.5, len(value_df.columns), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(value_df.index), 1), minor=True)
    ax.grid(which="minor", color="white", linewidth=1.5)
    ax.tick_params(which="minor", bottom=False, left=False)
    fig.tight_layout()
    return fig, ax

fig2, ax2 = plot_badness_heatmap(
    fosi_df,
    "A2: Sensitivity tests, FOSI -- configuration x metric\n"
    "color = relative badness within this table (lightest = best, darkest = worst per column); blank = not measured",
)
plt.show()